## Analysis of Clinical Determinants of Mortality in Breast Cancer

**Overview:**
This project investigates the clinical determinants of mortality in breast cancer patients. Utilizing Pandas, Seaborn, and Matplotlib, a custom statistical pipeline was engineered to conduct univariate exploratory data analysis across 13 clinical factors. By coupling visual distributions with rigorous mathematical validation (including Chi-Square, ANOVA, Welch's T-Test, and Multivariate Log-Rank tests), this analysis identifies the statistically significant prognostic drivers of patient survival.

-----------------------
### Notebook 1: Data Cleaning and Preprocessing
In this notebook, we look into the dataset (SEER Breast Cancer Dataset) for the first time, clean the dataset and reformat the dataset to ensure uniformity. Lastly, we setup a data dictionary for further reference.

In [1]:
import pandas as pd

In [2]:
# Import and load the dataset
df = pd.read_csv('../data/raw/SEER_Breast_Cancer_Dataset.csv')

In [3]:
# Inspect the first few rows to verify successful data loading
df.head()

,Age,Race,Marital Status,Unnamed: 3,T Stage,N Stage,6th Stage,Grade,A Stage,Tumor Size,Estrogen Status,Progesterone Status,Regional Node Examined,Reginol Node Positive,Survival Months,Status
0,43,"Other (American Indian/AK Native, Asian/Pacifi...",Married (including common law),NaN,T2,N3,IIIC,Moderately differentiated; Grade II,Regional,40,Positive,Positive,19,11,1,Alive
1,47,"Other (American Indian/AK Native, Asian/Pacifi...",Married (including common law),NaN,T2,N2,IIIA,Moderately differentiated; Grade II,Regional,45,Positive,Positive,25,9,2,Alive
2,67,White,Married (including common law),NaN,T2,N1,IIB,Poorly differentiated; Grade III,Regional,25,Positive,Positive,4,1,2,Dead
3,46,White,Divorced,NaN,T1,N1,IIA,Moderately differentiated; Grade II,Regional,19,Positive,Positive,26,1,2,Dead
4,63,White,Married (including common law),NaN,T2,N2,IIIA,Moderately differentiated; Grade II,Regional,35,Positive,Positive,21,5,3,Dead


In [4]:
# Validate information of the dataset: Number of entries, column names, non-null count and data type
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4024 entries, 0 to 4023
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Age                     4024 non-null   int64  
 1   Race                    4024 non-null   object 
 2   Marital Status          4024 non-null   object 
 3   Unnamed: 3              0 non-null      float64
 4   T Stage                 4024 non-null   object 
 5   N Stage                 4024 non-null   object 
 6   6th Stage               4024 non-null   object 
 7   Grade                   4024 non-null   object 
 8   A Stage                 4024 non-null   object 
 9   Tumor Size              4024 non-null   int64  
 10  Estrogen Status         4024 non-null   object 
 11  Progesterone Status     4024 non-null   object 
 12  Regional Node Examined  4024 non-null   int64  
 13  Reginol Node Positive   4024 non-null   int64  
 14  Survival Months         4024 non-null   

In [5]:
# Extract column names into a list to identify hidden trailing spaces or typos
df.columns.tolist()

['Age',
 'Race ',
 'Marital Status',
 'Unnamed: 3',
 'T Stage ',
 'N Stage',
 '6th Stage',
 'Grade',
 'A Stage',
 'Tumor Size',
 'Estrogen Status',
 'Progesterone Status',
 'Regional Node Examined',
 'Reginol Node Positive',
 'Survival Months',
 'Status']

In [6]:
# Drop the empty column
df = df.drop('Unnamed: 3', axis=1)
df.head()

,Age,Race,Marital Status,T Stage,N Stage,6th Stage,Grade,A Stage,Tumor Size,Estrogen Status,Progesterone Status,Regional Node Examined,Reginol Node Positive,Survival Months,Status
0,43,"Other (American Indian/AK Native, Asian/Pacifi...",Married (including common law),T2,N3,IIIC,Moderately differentiated; Grade II,Regional,40,Positive,Positive,19,11,1,Alive
1,47,"Other (American Indian/AK Native, Asian/Pacifi...",Married (including common law),T2,N2,IIIA,Moderately differentiated; Grade II,Regional,45,Positive,Positive,25,9,2,Alive
2,67,White,Married (including common law),T2,N1,IIB,Poorly differentiated; Grade III,Regional,25,Positive,Positive,4,1,2,Dead
3,46,White,Divorced,T1,N1,IIA,Moderately differentiated; Grade II,Regional,19,Positive,Positive,26,1,2,Dead
4,63,White,Married (including common law),T2,N2,IIIA,Moderately differentiated; Grade II,Regional,35,Positive,Positive,21,5,3,Dead


In [7]:
# Rename column with typo
df = df.rename(columns={'Reginol Node Positive':'Regional Node Positive'})

# Rename columns with spacing at the back
df = df.rename(columns={'Race ':'Race'})
df = df.rename(columns={'T Stage ':'T Stage'})

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4024 entries, 0 to 4023
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Age                     4024 non-null   int64 
 1   Race                    4024 non-null   object
 2   Marital Status          4024 non-null   object
 3   T Stage                 4024 non-null   object
 4   N Stage                 4024 non-null   object
 5   6th Stage               4024 non-null   object
 6   Grade                   4024 non-null   object
 7   A Stage                 4024 non-null   object
 8   Tumor Size              4024 non-null   int64 
 9   Estrogen Status         4024 non-null   object
 10  Progesterone Status     4024 non-null   object
 11  Regional Node Examined  4024 non-null   int64 
 12  Regional Node Positive  4024 non-null   int64 
 13  Survival Months         4024 non-null   int64 
 14  Status                  4024 non-null   object
dtypes: i

In [9]:
# Convert all string columns to title case to prevent split categories
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.title().str.strip()

# Convert '6th Stage' to uppercase
df['6th Stage'] = df['6th Stage'].str.upper().str.strip()

# Fix the Roman numerics in 'Grade'
df['Grade'] = df['Grade'].str.replace('Iii', 'III', regex=False).str.replace('Ii', 'II', regex=False).str.replace('Iv', 'IV', regex=False)

df.head()

,Age,Race,Marital Status,T Stage,N Stage,6th Stage,Grade,A Stage,Tumor Size,Estrogen Status,Progesterone Status,Regional Node Examined,Regional Node Positive,Survival Months,Status
0,43,"Other (American Indian/Ak Native, Asian/Pacifi...",Married (Including Common Law),T2,N3,IIIC,Moderately Differentiated; Grade II,Regional,40,Positive,Positive,19,11,1,Alive
1,47,"Other (American Indian/Ak Native, Asian/Pacifi...",Married (Including Common Law),T2,N2,IIIA,Moderately Differentiated; Grade II,Regional,45,Positive,Positive,25,9,2,Alive
2,67,White,Married (Including Common Law),T2,N1,IIB,Poorly Differentiated; Grade III,Regional,25,Positive,Positive,4,1,2,Dead
3,46,White,Divorced,T1,N1,IIA,Moderately Differentiated; Grade II,Regional,19,Positive,Positive,26,1,2,Dead
4,63,White,Married (Including Common Law),T2,N2,IIIA,Moderately Differentiated; Grade II,Regional,35,Positive,Positive,21,5,3,Dead


In [10]:
# Check for NULL values
df.isna().sum()

Age                       0
Race                      0
Marital Status            0
T Stage                   0
N Stage                   0
6th Stage                 0
Grade                     0
A Stage                   0
Tumor Size                0
Estrogen Status           0
Progesterone Status       0
Regional Node Examined    0
Regional Node Positive    0
Survival Months           0
Status                    0
dtype: int64

In [11]:
# Check the dimensionality of the dataset
df.shape

(4024, 15)

In [12]:
# Check for duplicate entries
duplicates = df[df.duplicated()]
duplicates

,Age,Race,Marital Status,T Stage,N Stage,6th Stage,Grade,A Stage,Tumor Size,Estrogen Status,Progesterone Status,Regional Node Examined,Regional Node Positive,Survival Months,Status
1010,63,White,Married (Including Common Law),T1,N1,IIA,Moderately Differentiated; Grade II,Regional,17,Positive,Positive,9,1,56,Alive


In [13]:
# Drop the duplicated entry
df.drop_duplicates(inplace=True)

# Check the new dimensionality of the dataset
df.shape

(4023, 15)

In [14]:
# Check if there are any errors in the dataset (eg: negatives in age/tumor size, etc.)
df.describe()

,Age,Tumor Size,Regional Node Examined,Regional Node Positive,Survival Months
count,4023.000000,4023.000000,4023.000000,4023.000000,4023.000000
mean,53.969923,30.477007,14.358439,4.158837,71.301765
std,8.963118,21.121253,8.100241,5.109724,22.923009
min,30.000000,1.000000,1.000000,1.000000,1.000000
25%,47.000000,16.000000,9.000000,1.000000,56.000000
50%,54.000000,25.000000,14.000000,2.000000,73.000000
75%,61.000000,38.000000,19.000000,5.000000,90.000000
max,69.000000,140.000000,61.000000,46.000000,107.000000


In [15]:
# Extract columns with categorical data
categorical_columns = df.select_dtypes(include=['object']).columns

# Extract unique data in each columns
for col in categorical_columns:
    unique_options = df[col].unique().tolist()
    print(f" - Column Name: {col}")
    print(f"   Options ({len(unique_options)} total): {unique_options}")
    print("-" * 45)

 - Column Name: Race
   Options (3 total): ['Other (American Indian/Ak Native, Asian/Pacific Islander)', 'White', 'Black']
---------------------------------------------
 - Column Name: Marital Status
   Options (5 total): ['Married (Including Common Law)', 'Divorced', 'Single (Never Married)', 'Widowed', 'Separated']
---------------------------------------------
 - Column Name: T Stage
   Options (4 total): ['T2', 'T1', 'T3', 'T4']
---------------------------------------------
 - Column Name: N Stage
   Options (3 total): ['N3', 'N2', 'N1']
---------------------------------------------
 - Column Name: 6th Stage
   Options (5 total): ['IIIC', 'IIIA', 'IIB', 'IIA', 'IIIB']
---------------------------------------------
 - Column Name: Grade
   Options (4 total): ['Moderately Differentiated; Grade II', 'Poorly Differentiated; Grade III', 'Well Differentiated; Grade I', 'Undifferentiated; Anaplastic; Grade IV']
---------------------------------------------
 - Column Name: A Stage
   Options

In [16]:
# Save the clean dataset
df.to_csv('../data/processed/data_clean.csv', index=False)

#### Setup a data dictionary

**Data Dictionary:**

| No. | Variable               | Description                                                            | Type        |
|:----|:-----------------------|:-----------------------------------------------------------------------|:------------|
| 0   | Age                    | Age of patient at diagnosis                                            | Numeric     |
| 1   | Race                   | Race of patient                                                        | Categorical |
| 2   | Marital status         | Marital status of patient                                              | Categorical |
| 3   | T Stage                | Stage of cancer according to tumor size                                | Categorical |
| 4   | N Stage                | Stage of cancer according to the spreading to lymph nodes              | Categorical |
| 5   | 6th Stage              | Overall stage of cancer                                                | Categorical |
| 6   | Grade                  | Grade of cancer, refers to appearance and behavior of the cancer cells | Categorical |
| 7   | A Stage                | Distant metastasis status                                              | Categorical |
| 8   | Tumor Size             | Size of tumor at diagnosis (millimeter(mm))                            | Numeric     |
| 9   | Estrogen Status        | Estrogen receptor status                                               | Categorical |
| 10  | Progesterone Status    | Progesterone receptor status                                           | Categorical |
| 11  | Regional Node Examined | Total count of regional lymph nodes examined                           | Numeric     |
| 12  | Regional Node Positive | Total count of regional lymph nodes positive for cancer                | Numeric     |
| 13  | Survival Months        | Number of months patient survived after diagnosis                      | Numeric     |
| 14  | Status                 | Whether patient is alive or dead                                       | Categorical |










